# Load Package

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

# Import Functions
sys.path.append("../../")

from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from os.path import join, exists
import sys
sys.path.append("../") # Add directory containing src/data to path

from src.configs.blood_config import data_name, batch_size, eval_batch_size, data_name_ood_in, data_name_ood_out, data_name_ood_diff
from src.configs.default_configs import fn_pred
from src.file_manager.filepath import FilePath
from src.file_manager.load_save_df import load_pred_df, save_pred_perf_df
from src.evaluation.evaluate import get_model_performance
from src.file_manager.load_save_model import load_model

from src.models.bnn.model import BNN
from src.models.bnn.train import train_bnn_w_best_param
from src.models.bnn.predict import get_bnn_model_prediction
from src.training.misc import get_class_weights
from src.evaluation.inference import get_all_predictions, split_test_set
from src.file_manager.load_save_df import save_pred_df, load_pred_df

from src.data_generator.blood import load_bloodmnist_data_dict
from src.data_generator.raabin import load_raabin_data_dict
from src.data_generator.bonemarrow import load_bonemarrow_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets

from cur_seed import seed
cur_model_name="tuned"
# seed = 2024


fp = FilePath(data_name=data_name, seed=seed)
fp_ood_in = FilePath(data_name=data_name_ood_in, seed=seed)
fp_ood_out = FilePath(data_name=data_name_ood_out, seed=seed)
fp_ood_diff = FilePath(data_name=data_name_ood_diff, seed=seed)

# Load Data

In [ ]:
print("Loading Data Dict")
data_dict = load_bloodmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])

# Add ID Class OOD into the Test Set
print("Loading ID Class OOD Data Dict")
data_dict_ood_in = load_raabin_data_dict(fp_preprocessed=fp_ood_in.get_preprocessed_folder())
data_dict_ood_in = process_dataset_for_ood(data_dict, data_dict_ood_in, seed)

# ODD Class OOD
print("Loading OOD Class OOD Data Dict")
data_dict_ood_out = load_bonemarrow_data_dict(fp_preprocessed=fp_ood_out.get_preprocessed_folder())
data_dict_ood_out = process_dataset_for_ood(data_dict, data_dict_ood_out, seed)

# OOD Modality
print("Loading OOD Modality OOD Data Dict")
data_dict_ood_diff = load_chestmnist_data_dict(
    fp_preprocessed=fp_ood_diff.get_preprocessed_folder(), only_test=True, num_classes=8)
data_dict_ood_diff = process_dataset_for_ood(data_dict, data_dict_ood_diff, seed)

data_dict = left_join_datasets(data_dict, data_dict_ood_in)

# Training

In [ ]:
if not exists(fp.get_fp_model(BNN, cur_model_name="tuned")):
    best_param = {"feat_extractor":"resnet"}
    class_weights = get_class_weights(data_dict)
    bnn_model = train_bnn_w_best_param(
        best_param, data_dict=data_dict, 
        epochs=500, patience=5, seed=seed, 
        fp=fp, class_weight=class_weights,
        batch_size=batch_size, eval_batch_size=batch_size, 
    )

# Prediction

In [ ]:
if not exists(join(fp.get_fp_df(BNN, fn_pred))):
    bnn_model = load_model(fp=fp, ModelClass=BNN, cur_model_name=cur_model_name)
    pred_df = get_all_predictions(
        model=bnn_model, 
        data_dict=data_dict, 
        batch_size=batch_size, 
        eval_batch_size=eval_batch_size,
        pred_func=get_bnn_model_prediction,
        seed=seed,
        additional_pred_args={"T":10}
    )
    pred_df["split_perf"] = pred_df["split"]
    pred_df = split_test_set(
            pred_df, split_col="split", new_split_col="split_perf", 
            num_ori_test=num_ori_test, labels=["Test-Blood", "Test-Raabin"])
    save_pred_df(pred_df=pred_df, fp=fp, ModelClass=BNN)


# Performance Evaluation

In [ ]:
pred_df = load_pred_df(fp=fp, ModelClass=BNN)
perf_df = get_model_performance(
    all_pred_df=pred_df, data_dict=data_dict, label="bnn", perf_split_col="split_perf")
save_pred_perf_df(pred_perf_df=perf_df, fp=fp, ModelClass=BNN)
perf_df

# OOD Prediction

In [ ]:
bnn_model = load_model(fp=fp, ModelClass=BNN, cur_model_name=cur_model_name)
ood_dicts = {
    "ood_in_raabin": data_dict_ood_in, 
    "ood_out_bonemarrow": data_dict_ood_out,
    "ood_chestmnist": data_dict_ood_diff}
for label, cur_ood_data_dict in tqdm(ood_dicts.items(), total=len(ood_dicts)):
    pred_df_ood = get_all_predictions(
        model=bnn_model, 
        data_dict=cur_ood_data_dict, 
        batch_size=batch_size, 
        eval_batch_size=eval_batch_size,
        pred_func=get_bnn_model_prediction,
        seed=seed,
        additional_pred_args={"T":10, "ood": True},
    )
    save_pred_df(pred_df=pred_df_ood, fp=fp, ModelClass=BNN, optional_label=label)